# Load and sample from the AICME runtime bundle

This notebook shows the consumer-facing runtime path exported by `load_and_push.ipynb`.

Workflow:
1. Load the Hugging Face runtime bundle `cesarali/AICME-runtime` with `transformers`.
2. Read the local example study from `data/preprocessed/Theophylline.json`.
3. Inspect the runtime builder capacity from `model.config.builder_config` and trim only when required.
4. Generate new target individuals with `run_task(task="generate", ...)`.

The notebook keeps the inference path consumer-oriented: it does not import the local `pff` package for model loading.

In [2]:
INSTALL_RUNTIME_DEPS = True

if INSTALL_RUNTIME_DEPS:
    import subprocess
    import sys

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch",
            "transformers",
            "huggingface_hub",
            "lightning",
            "datasets",
            "pandas",
            "matplotlib",
            "torchtyping",
            "gpytorch",
            "pot",
            "torchdiffeq",
            "torchsde",
            "ruamel.yaml",
            "pyyaml",
        ]
    )


  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached torchtyping-0.1.5-py3-none-any.whl.metadata (9.5 kB)
  Using cached pot-0.9.6.post1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached torchdiffeq-0.2.5-py3-none-any.whl.metadata (440 bytes)
  Using cached torchsde-0.2.6-py3-none-any.whl.metadata (5.3 kB)
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached xxhash-3.6.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached typeguard-2.13.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2

In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from transformers import AutoModel


def find_repo_root(start: Path) -> Path:
    """Locate the repository root from the current working directory."""

    resolved_start = start.resolve()
    for candidate in [resolved_start, *resolved_start.parents]:
        if (candidate / "pff").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. Start the notebook from somewhere inside pff."
    )


REPO_ROOT = find_repo_root(Path.cwd())
REPO_ROOT


In [ ]:
RUNTIME_REPO_ID = "cesarali/AICME-runtime"
NUM_SAMPLES = 6

model = AutoModel.from_pretrained(
    RUNTIME_REPO_ID,
    trust_remote_code=True,
)
model.eval()

builder_config = dict(model.config.builder_config)
pd.Series(builder_config, name="runtime_capacity")


## Load and prepare the Theophylline study

The runtime contract rejects studies that exceed the stored builder capacity. `Theophylline.json` contains 12 context individuals, so this notebook trims the study automatically if the loaded runtime advertises a smaller `max_context_individuals` value.

In [ ]:
def trim_individual(individual, *, max_observations: int, max_remaining: int):
    """Trim one individual to the runtime observation capacities."""

    trimmed = deepcopy(individual)

    observations = list(trimmed.get("observations", []))
    observation_times = list(trimmed.get("observation_times", []))
    if len(observations) > max_observations:
        trimmed["observations"] = observations[:max_observations]
        trimmed["observation_times"] = observation_times[:max_observations]

    remaining = list(trimmed.get("remaining", []))
    remaining_times = list(trimmed.get("remaining_times", []))
    if len(remaining) > max_remaining:
        trimmed["remaining"] = remaining[:max_remaining]
        trimmed["remaining_times"] = remaining_times[:max_remaining]

    return trimmed


def prepare_study_for_runtime(study, builder_cfg):
    """Trim study blocks so they satisfy the runtime bundle contract."""

    prepared = deepcopy(study)

    original_context_count = len(prepared.get("context", []))
    original_target_count = len(prepared.get("target", []))

    prepared["context"] = [
        trim_individual(
            individual,
            max_observations=int(builder_cfg["max_context_observations"]),
            max_remaining=int(builder_cfg["max_context_remaining"]),
        )
        for individual in prepared.get("context", [])[: int(builder_cfg["max_context_individuals"])]
    ]
    prepared["target"] = [
        trim_individual(
            individual,
            max_observations=int(builder_cfg["max_target_observations"]),
            max_remaining=int(builder_cfg["max_target_remaining"]),
        )
        for individual in prepared.get("target", [])[: int(builder_cfg["max_target_individuals"])]
    ]

    report = {
        "study_name": prepared.get("meta_data", {}).get("study_name", ""),
        "substance_name": prepared.get("meta_data", {}).get("substance_name", ""),
        "original_context_individuals": original_context_count,
        "used_context_individuals": len(prepared.get("context", [])),
        "original_target_individuals": original_target_count,
        "used_target_individuals": len(prepared.get("target", [])),
        "was_trimmed": (
            original_context_count != len(prepared.get("context", []))
            or original_target_count != len(prepared.get("target", []))
        ),
    }
    return prepared, report


study_path = REPO_ROOT / "data" / "preprocessed" / "Theophylline.json"
studies = json.loads(study_path.read_text(encoding="utf-8"))
raw_study = studies[0]
prepared_study, capacity_report = prepare_study_for_runtime(raw_study, builder_config)

pd.DataFrame([capacity_report])


In [ ]:
outputs = model.run_task(
    task="generate",
    studies=[prepared_study],
    num_samples=NUM_SAMPLES,
)

generated_samples = outputs["results"][0]["samples"]

sample_summary = pd.DataFrame(
    [
        {
            "sample_idx": sample_idx,
            "target_name_id": sample["target"][0].get("name_id", f"sample_{sample_idx}"),
            "num_observations": len(sample["target"][0].get("observations", [])),
            "time_min": min(sample["target"][0].get("observation_times", [0.0])),
            "time_max": max(sample["target"][0].get("observation_times", [0.0])),
        }
        for sample_idx, sample in enumerate(generated_samples)
    ]
)

display(pd.Series(outputs["model_info"], name="model_info"))
sample_summary


In [ ]:
PLOT_LOG_SCALE = False

fig, ax = plt.subplots(figsize=(8, 5))

for context_individual in prepared_study["context"]:
    ax.plot(
        context_individual["observation_times"],
        context_individual["observations"],
        color="0.75",
        alpha=0.6,
        linewidth=1.0,
    )

for sample_idx, sample in enumerate(generated_samples):
    target_individual = sample["target"][0]
    ax.plot(
        target_individual["observation_times"],
        target_individual["observations"],
        marker="o",
        linewidth=2.0,
        label=f"generated sample {sample_idx}",
    )

if PLOT_LOG_SCALE:
    ax.set_yscale("log")

ax.set_title(f"{prepared_study['meta_data']['substance_name']} runtime samples")
ax.set_xlabel("time")
ax.set_ylabel("concentration")
ax.grid(alpha=0.25)
ax.legend(ncols=2, fontsize=9)
plt.show()


In [ ]:
generated_samples[0]
